In [ ]:
!curl -LO https://raw.githubusercontent.com/MohamadMerchant/SNLI/master/data.tar.gz
!tar -xvzf data.tar.gz
!pip install loguru

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11.1M  100 11.1M    0     0  11.5M      0 --:--:-- --:--:-- --:--:-- 11.5M
SNLI_Corpus/
SNLI_Corpus/snli_1.0_dev.csv
SNLI_Corpus/snli_1.0_train.csv
SNLI_Corpus/snli_1.0_test.csv


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import transformers
from sklearn.manifold import TSNE
from loguru import logger
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim

# 1) HuggingFace - Text Classification

## 1.1 Data Import

In [ ]:
# Load the training dataset from a CSV file, limiting to 100,000 rows.
# Output: (num_samples, num_features) (DataFrame)
train_df = pd.read_csv("./SNLI_Corpus/snli_1.0_train.csv", nrows=100000)

# Load the validation dataset from a CSV file.
# Output: (num_samples, num_features) (DataFrame)
valid_df = pd.read_csv("./SNLI_Corpus/snli_1.0_dev.csv")

# Load the test dataset from a CSV file.
# Output: (num_samples, num_features) (DataFrame)
test_df = pd.read_csv("./SNLI_Corpus/snli_1.0_test.csv")

# Data Processing
train_df = (
    train_df[train_df.similarity != "-"]
    .sample(frac=1.0, random_state=42).reset_index(drop=True)
)

valid_df = (
    valid_df[valid_df.similarity != "-"]
    .sample(frac=1.0, random_state=42).reset_index(drop=True)
)

train_df.head()

,similarity,sentence1,sentence2
0,contradiction,A woman is using toy which blows giant bubbles.,A little girl is playing with chalk on a drive...
1,neutral,A young Asian girl holds a stuffed cat toy in ...,A young Asian girl sits in class with a stuffe...
2,entailment,A young woman with an afro and an electronic d...,A young woman walks next to an orange bike.
3,neutral,A young asian girl is sliding down a pole on o...,The girl has yellow skin
4,entailment,a man is walking with a cane.,The man is walking.


In [ ]:
# Map unique similarity values to numerical labels.
# Output: (num_classes,) (dict)
label_map = dict(enumerate(train_df['similarity'].astype('category').cat.categories))

# Encode the 'similarity' column
# Output: (num_samples,) (ndarray of ints)
y_train = train_df['similarity'].map({v:k for k, v in label_map.items()}).values
y_val = valid_df['similarity'].map({v:k for k, v in label_map.items()}).values
y_test = test_df['similarity'].map({v:k for k, v in label_map.items()}).values

## 1.2 Pre-processing

In [ ]:
max_length = 64
batch_size = 32

In [ ]:
# Initialize the BERT tokenizer from a pre-trained 'bert-base-uncased' model.
# The tokenizer will convert text into numerical IDs suitable for BERT models.
tokenizer = transformers.BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)

# Print the size of the tokenizer's vocabulary.
print(len(tokenizer.get_vocab()))
tokenizer.encode('Hello Tensorflow')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

30522


[101, 7592, 23435, 12314, 102]

In [ ]:
# Input: (batch_num, 2)
# Output: (batch_num, max_len) (PyTorch Tensors)
sentence_pairs = train_df[["sentence1", "sentence2"]].values[:5]
encoded = tokenizer.batch_encode_plus(
    sentence_pairs.tolist(),
    add_special_tokens=True,
    max_length=max_length,
    return_attention_mask=True,
    return_token_type_ids=True,
    padding='max_length',
    return_tensors="pt")

# Display the structure and contents of the encoded batch.
print(encoded.keys())
print(encoded['input_ids'][0][:32])
print(encoded['token_type_ids'][0][:32])
print(encoded['attention_mask'][0][:32])

# Decode the first sequence of tokens back into human-readable text.
print(tokenizer.decode(encoded['input_ids'][0][:32]))

KeysView({'input_ids': tensor([[  101,  1037,  2450,  2003,  2478,  9121,  2029, 13783,  5016, 17255,
          1012,   102,  1037,  2210,  2611,  2003,  2652,  2007, 16833,  2006,
          1037, 11202,  1012,   102,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0],
        [  101,  1037,  2402,  4004,  2611,  4324,  1037, 11812,  4937,  9121,
          1999,  1037,  9823,  1012,   102,  1037,  2402,  4004,  2611,  7719,
          1999,  2465,  2007,  1037, 11812,  4937,  9121,  1010,  1996,  2069,
          6405,  6664,  3588,  2044,  1996, 19267,  1012,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0

In [ ]:
class BertDataGenerator(torch.utils.data.Dataset):
    """Generates batches of data.

    Args:
        sentence_pairs: Array of premise and hypothesis input sentences.
        labels: Array of labels.
        shuffle: boolean, whether to shuffle the data.
        include_targets: boolean, whether to incude the labels.

    Returns:
        Tuples `([input_ids, attention_mask, `token_type_ids], labels)`
        (or just `[input_ids, attention_mask, `token_type_ids]`
         if `include_targets=False`)
    """

    def __init__(self, sentence_pairs, labels, shuffle=True, include_targets=True):

        self.sentence_pairs = sentence_pairs
        self.labels = labels
        self.shuffle = shuffle
        self.include_targets = include_targets
        self.tokenizer = transformers.BertTokenizer.from_pretrained( "bert-base-uncased", do_lower_case=True)
        self.indexes = np.arange(len(self.sentence_pairs))
        self.on_epoch_end()

    def __len__(self):
        # Denotes the number of batches per epoch.
        return len(self.sentence_pairs)

    def __getitem__(self, idx):
        # Retrieves the batch of index.
        sentence_pair = self.sentence_pairs[self.indexes[idx]]

        # batch_encode_plus return dict (input, atttention_mask, token_type)
        # encode only return list of input_ids
        # encoded together and separated by [SEP] token.
        encoded = self.tokenizer.encode_plus(
            sentence_pair[0],
            sentence_pair[1],
            add_special_tokens=True,
            truncation=True,
            max_length=max_length,
            return_attention_mask=True,
            return_token_type_ids=True,
            padding='max_length',
            return_tensors="pt",
        )

        # Convert batch of encoded features to numpy array.
        # (seq_len,)
        input_ids = encoded["input_ids"].squeeze(0)
        attention_masks = encoded["attention_mask"].squeeze(0)
        token_type_ids = encoded["token_type_ids"].squeeze(0)

        # Set to true if data generator is used for training/validation.
        if self.include_targets:
            label = torch.tensor(self.labels[self.indexes[idx]], dtype=torch.long)
            return [input_ids, attention_masks, token_type_ids], label
        else:
            return [input_ids, attention_masks, token_type_ids]

    def on_epoch_end(self):
        # Shuffle indexes after each epoch if shuffle is set to True.
        if self.shuffle:
            np.random.RandomState(42).shuffle(self.indexes)

## 1.3 Model Building

Outputs of Bert Model comprised of:

- ***last_hidden_state*** with shape=(batch_size, seq_len, embed_dim). The first token of every last_hidden_state is the special classification token – [CLS]. This token is used in classification tasks as an aggregate of the entire sequence representation. It is ignored in non-classification tasks.

- ***pooler_output*** represent each input sequence as a whole with shape=(batch_size, emb_dim). Last layer hidden-state of the first token of the sequence [CLS] further processed by a Linear layer and a Tanh activation function.

- ***hidden_states*** which generate the hidden state for all transformer layers. Only when set *output_hidden_states=True*, shape=(num_layers, batch_size, seq_len, emb_dim).  In a 4-layers BERT model a token will have 4 intermediate representations. The last value of the list is equal to **last_hidden_state**.

- ***attentions*** attention weights from each layer.  Only when set *output_attentions=True*, shape=(num_layers, batch_size, seq_len, emb_dim)




In [ ]:
class BertClassifier(nn.Module):
    def __init__(self, num_classes):
        super(BertClassifier, self).__init__()
        # Initialize the pre-trained BERT model
        self.bert = transformers.BertModel.from_pretrained("bert-base-uncased")
        # Dropout layer to prevent overfitting
        self.dropout = nn.Dropout(0.3)
        # Linear layer to project BERT output to class logits
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)
        # ReLU activation function
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, token_type_ids):
        # Pass inputs through the BERT encoder
        # Input: (batch_num, max_len)
        # last_hidden_state (batch_num, max_len, embed_dim)
        # pooler_output (batch_num, embed_dim)
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # Extract the [CLS] token representation after a linear layer and tanh
        # Apply dropout to the pooled representation
        # Output: (batch_num, embed_dim)
        pooled_output = outputs.pooler_output
        dropout_output = self.dropout(pooled_output)

        # Project to class logits
        # Input: (batch_num, embed_dim)
        # Output: (batch_num, num_classes)
        logits = self.fc(dropout_output)

        return logits

# Determine number of output classes
num_classes = len(label_map)
# Instantiate the classifier
model = BertClassifier(num_classes)

## 1.4 Model Training (Frozen Pre-trained Bert)

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn

# Initialize data generators for training and validation
train_data = BertDataGenerator(
    train_df[["sentence1", "sentence2"]].values.astype("str"),
    y_train,
    shuffle=True
)
valid_data = BertDataGenerator(
    valid_df[["sentence1", "sentence2"]].values.astype("str"),
    y_val,
    shuffle=False
)

# Create DataLoaders to handle batching
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_data, batch_size=batch_size, shuffle=False)

# Define loss function
criterion = nn.CrossEntropyLoss()

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def unpack_batch(inputs, labels):
    """Move a batch of inputs and labels to the target device.
    # Input: list of tensors (batch_num, max_len), labels (batch_num)
    # Output: 4 tensors on device — input_ids (batch_num, max_len),
    #         attention_mask (batch_num, max_len), token_type_ids (batch_num, max_len),
    #         labels (batch_num)
    """
    input_ids, attention_mask, token_type_ids = inputs
    return (
        input_ids.to(device),
        attention_mask.to(device),
        token_type_ids.to(device),
        labels.to(device),
    )


def train_one_epoch(model, optimizer, epoch, total_epochs, prefix=""):
    """Run one full training pass over the training set."""
    model.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_dataloader):
        # Unpack and transfer batch to device
        # Input: (batch_num, max_len), labels (batch_num)
        input_ids, attention_mask, token_type_ids, labels = unpack_batch(inputs, labels)

        # Reset gradients from previous step
        optimizer.zero_grad()

        # Forward pass through the BERT classifier
        # Input: input_ids,  attention_mask, token_type_ids (batch_num, max_len)
        # Output: (batch_num, num_classes)
        outputs = model(input_ids, attention_mask, token_type_ids)

        # Compute cross-entropy loss between predictions and ground truth
        # Input: (batch_num, num_classes), (batch_num) → Output: scalar
        loss = criterion(outputs, labels)

        # Backward pass and parameter update
        loss.backward()
        optimizer.step()

        # Accumulate loss and log every 100 batches
        running_loss += loss.item()
        if i % 100 == 99:
            print(f'{prefix}Epoch [{epoch + 1}/{total_epochs}], '
                  f'Step [{i + 1}/{len(train_dataloader)}], '
                  f'Loss: {running_loss / 100:.4f}')
            running_loss = 0.0


def validate(model, epoch, total_epochs, prefix=""):
    """Evaluate the model on the validation set and print accuracy."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in valid_dataloader:
            # Unpack and transfer batch to device
            # Input: list of 3 tensors (batch_num, max_len), labels (batch_num)
            # Output: tensors on device with same shapes
            input_ids, attention_mask, token_type_ids, labels = unpack_batch(inputs, labels)

            # Forward pass (inference only, no gradient tracking)
            # Input: input_ids (batch_num, max_len), attention_mask (batch_num, max_len),
            #        token_type_ids (batch_num, max_len)
            # Output: (batch_num, num_classes)
            outputs = model(input_ids, attention_mask, token_type_ids)

            # Get predicted class indices
            # Input: (batch_num, num_classes) → Output: (batch_num)
            _, predicted = torch.max(outputs.data, 1)

            # Accumulate correct predictions
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'{prefix}Epoch [{epoch + 1}/{total_epochs}], '
          f'Validation Accuracy: {100 * correct / total:.2f}%')


def train(model, lr, epochs, freeze_bert=False, prefix=""):
    """Full training loop: optionally freeze BERT, then train and validate each epoch."""
    # Freeze or unfreeze BERT encoder parameters
    for param in model.bert.parameters():
        param.requires_grad = not freeze_bert

    # Initialize Adam optimizer with the specified learning rate
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        train_one_epoch(model, optimizer, epoch, epochs, prefix)
        validate(model, epoch, epochs, prefix)


# Phase 1 — Train only the classifier head with BERT layers frozen
print("Training with frozen BERT layers...")
train(model, lr=1e-5, epochs=2, freeze_bert=True)

# Phase 2 — Fine-tune all layers (BERT + classifier) with a lower learning rate
print("\nTraining with unfrozen BERT layers...")
train(model, lr=1e-6, epochs=1, freeze_bert=False, prefix="Fine-tune ")

print("Finished Training")